# 강의 13 - 에이전트 메모리


## 설정

이 노트북은 **Microsoft Agent Framework**(MAF)를 사용하여 <strong>지속적인 메모리</strong>를 갖춘 여행 예약 에이전트를 만드는 방법을 보여줍니다.

작업용, 단기 및 장기 메모리 등 다양한 유형의 에이전트 메모리가 대화 전반에 걸쳐 에이전트가 정보를 유지하고 사용하는 방식에 어떤 영향을 미치는지 배우게 됩니다.

**필수 조건:**
- 배포된 채팅 모델(예: `gpt-4.1-mini`)이 포함된 Microsoft Foundry 프로젝트.
- Azure CLI에 로그인 — 터미널에서 `az login` 명령 실행.
- `AZURE_AI_PROJECT_ENDPOINT` — Microsoft Foundry 프로젝트 엔드포인트.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 배포된 모델 이름.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## 에이전트 메모리의 유형

AI 에이전트는 각각 고유한 목적을 가진 다양한 종류의 메모리를 활용할 수 있습니다:

### 작업 메모리
대화 스레드 자체 — 단일 세션에서 주고받은 메시지들입니다. 에이전트는 일관성을 유지하기 위해 같은 스레드 내의 이전 메시지를 참조할 수 있습니다. MAF에서는 **`agent.create_session()`** 을 통해 생성되며, `AgentSession` 을 반환합니다.

### 단기 메모리
작업이나 세션 동안 지속되지만 영구적으로 저장되지는 않는 정보입니다. 예를 들어, 에이전트는 다회전 계획 대화 중에 사실들을 축적하고 이를 사용해 최종 일정을 생성할 수 있습니다.

### 장기 메모리
**세션 간에** 지속되는 선호도 및 사실들입니다. 돌아온 사용자는 식이 제한이나 여행 스타일을 반복해서 말할 필요가 없어야 합니다. 장기 메모리는 일반적으로 외부 저장소(데이터베이스, 파일, 벡터 인덱스 등)를 기반으로 하며, 도구를 통해 에이전트에 제공됩니다.


## 세션을 통한 작업 기억

가장 단순한 형태의 기억은 대화 세션입니다. 동일한 세션(`agent.create_session()`을 통해 생성된)을 연속적인 `agent.run()` 호출에 전달하면, 에이전트는 해당 대화의 전체 기록을 보고 이전 세부 정보를 기억할 수 있습니다.

여행 에이전트를 만들고 작업 기억을 시연해 봅시다.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

에이전트가 예산을 정확히 기억한 것은 두 메시지가 같은 세션을 공유하기 때문입니다. 이것이 바로 **작동 기억(working memory)** — 세션이 지속되는 동안에만 존재합니다.

### 새로운 스레드에서는 어떤 일이 발생할까요?

<strong>새로운</strong> 세션을 만들면, 에이전트는 이전 대화 내용을 기억하지 못합니다:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## 장기 메모리 패턴

사용자 선호도를 **세션 간에** 기억하기 위해서는 대화 스레드 밖에 존재하는 영속적인 저장소가 필요합니다. 에이전트는 이 저장소에 접근하기 위해 정보를 저장하고 검색할 수 있는 함수인 <strong>도구</strong>를 사용합니다.

아래는 간단한 인메모리 선호도 저장소를 구현한 예시입니다(실제로는 데이터베이스나 벡터 인덱스로 대체할 수 있음) 그리고 이를 에이전트가 사용할 수 있는 도구로 노출합니다.

### 아키텍처
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### 시나리오 1 — 처음 방문하는 사용자가 기념일 여행 예약하기

Sarah는 처음 방문합니다. 에이전트는 도구를 통해 그녀의 선호도를 저장하고 이를 사용해 호텔을 추천해야 합니다.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### 시나리오 2 — 사라는 몇 주 후에 돌아옵니다

사라는 <strong>새로운 스레드</strong>를 시작합니다(새 세션을 시뮬레이션). 작업 메모리는 비어 있지만, 장기 선호 저장소에는 여전히 그녀의 정보가 있습니다. 에이전트는 이를 검색하여 추천을 개인화하는 데 사용해야 합니다.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## 요약

이번 수업에서는 세 가지 유형의 에이전트 메모리와 Microsoft Agent Framework로 이를 구현하는 방법을 배웠습니다:

| 메모리 유형 | MAF 메커니즘 | 수명 |
|---|---|---|
| **작업 메모리** | `agent.create_session()` | 단일 대화 |
| **단기 메모리** | 스레드 내 누적된 컨텍스트 | 단일 작업/세션 |
| **장기 메모리** | `@tool` 함수로 접근하는 외부 저장소 | 세션 간 지속 |

### 주요 내용
1. **`agent.create_session()`**은 작업 메모리를 제공합니다 — 에이전트는 세션 내 전체 대화 기록을 확인할 수 있습니다.
2. **새 세션은 컨텍스트를 잃습니다** — 장기 메모리가 없으면 에이전트는 이전 대화를 기억할 수 없습니다.
3. **`@tool` 함수**가 격차를 연결합니다 — 에이전트가 영구 저장소에 정보를 저장하고 불러올 수 있게 합니다.
4. **개인화는 시간이 지날수록 개선됩니다** — 선호도가 더 많이 저장될수록 에이전트의 추천이 더 나아집니다.

### 실제 적용 사례
- **고객 서비스**: 고객 이력 및 선호도 기억
- **개인 비서**: 여러 날 또는 주에 걸쳐 컨텍스트 유지
- <strong>헬스케어</strong>: 환자 정보 및 선호도 추적
- <strong>전자상거래</strong>: 이력을 바탕으로 한 개인화 쇼핑

### 다음 단계
- 메모리 내 딕셔너리를 데이터베이스 또는 벡터 스토어(예: Azure AI Search)로 교체
- 시간 민감 정보에 대한 메모리 만료 추가
- 공유 메모리가 있는 다중 에이전트 시스템 구축
- 지식 그래프 기반 메모리를 위한 Cognee 노트북 탐색


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
